In [0]:
from pyspark.sql.functions import col, regexp_replace
import pandas as pd

In [0]:
dbutils.fs.unmount("/mnt/weather_data")

In [0]:
# Define variables
storage_account_name = "aimasterdata"
storage_account_key = "j0/fRQr9nKT5rozOswNOMIOWQIwyudJH5oDSXINkttZBTP9T1O8phxCx3bJlPjz/kG9gz5Rqd4oW+AStbqPmlw=="
container_name = "weatherdata"
mount_point = "/mnt/weather_data"

# Mount the storage account
dbutils.fs.mount(
    source=f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point=mount_point,
    extra_configs={f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_key}
)

# Verify mount
display(dbutils.fs.ls(mount_point))


In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("delimiter", ";") \
    .option("inferSchema", "true") \
    .load("/mnt/weather_data/subs.csv")

df.display()


In [0]:
df = df.withColumnRenamed("SNIRH - SISTEMA NACIONAL DE INFORMA��O DE RECURSOS H�DRICOS", "_c0")

In [0]:
# Extract headers
header_stations = df.collect()[1]  # Location names
header_variables = df.collect()[2]  # Measurement type

# Ensure Date column is included
columns_to_keep = [0]  # Assuming Date is always the first column
cleaned_columns = ["Date"]  # First column must be "Date"

# Process remaining columns, skipping "FLAG"
for i in range(0, len(header_stations)):  # Start from index 1 (skip "Date" column)
    station = header_stations[i]
    variable = header_variables[i]

    if station and variable and "FLAG" not in variable:
        columns_to_keep.append(i)  # Store valid column index
        cleaned_columns.append(f"{station.strip()}_{variable.strip()}")  # Format column name


# Print debug info
print(f"Final columns_to_keep: {len(columns_to_keep)}")
print(f"Final cleaned_columns: {len(cleaned_columns)}")

In [0]:

# Select only relevant columns
df_filtered = df.select([col(f"_c{i}") for i in columns_to_keep])

# Rename columns
df_cleaned = df_filtered.toDF(*cleaned_columns)

df_cleaned.display()

In [0]:
print(f"Columns detected in dataset: {len(columns_to_keep)}")
print(f"Columns assigned for renaming: {len(cleaned_columns)}")

In [0]:
# Extract station names by removing the measurement type (e.g., "_Temperature")
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Print debug info
print(f"Total unique weather stations found: {len(stations)}")


In [0]:
dbutils.fs.rm("/mnt/weather_data/split_stations/", True)


In [0]:
pip install unidecode

In [0]:
cleaned_columns

In [0]:
stations

In [0]:
# Extract station names by removing the measurement type (e.g., "_Temperature")
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Print debug info
print(f"Total unique weather stations found: {len(stations)}")

In [0]:
for station in stations:
    
    print(f"✅ Done {station}")  # May or may not appear!

In [0]:
cleaned_columns

In [0]:
from unidecode import unidecode
import re

# Define output directory in Databricks FileStore or mounted path
output_dir = "/mnt/weather_data/split_stations/"

# Extract station names based on column prefixes
stations = list(set([col.split("_")[0] for col in cleaned_columns if "_" in col]))

# Loop through each station and write a separate CSV
for station in stations:
    # Get all columns for this station
    station_columns = ["Date"] + [col for col in df_cleaned.columns if col.startswith(station)]

    # Create sub-DataFrame for this station
    df_station = df_cleaned.select(*station_columns)

    # Normalize station name
    station_clean = unidecode(station)
    station_clean = re.sub(r"\s*\(.*?\)", "", station_clean)  # Remove content in parentheses
    station_clean = station_clean.replace(" ", "_")

    # Output path
    output_path = f"{output_dir}{station_clean}.csv"

    # Save to CSV
    df_station.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(output_path)

    print(f"✅ Saved: {output_path}")


In [0]:
station_columns

In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_stations/"))


In [0]:
# Example: Read one station's CSV file (replace with an actual filename from the previous step)
file_to_check = "/mnt/weather_data/split_stations/PROENA-A-NOVA.csv"

df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_to_check)

# Display the file contents
df_check.display()

In [0]:
# Define the base directory
base_dir = "/mnt/weather_data/split_stations/"

# Normalize location names (remove accents, fix underscores, and remove extra parts)
def normalize_name(name):
    name_clean = unidecode(name)  # Remove accents
    name_clean = re.sub(r"\s*\(.*?\)", "", name_clean)  # Remove everything inside parentheses
    name_clean = name_clean.replace(" ", "_")  # Replace spaces with underscores
    return name_clean

# Normalize all location names
locations_normalized = set([normalize_name(loc) for loc in stations])

print(f"✅ Using {len(locations_normalized)} locations after normalization.")


In [0]:
from pyspark.sql.functions import col

# Get a list of correct locations
locations = list(locations_normalized)

for location in locations:
    print(f"🔹 Merging data for {location}...")

    # Define the corrected file paths
    temp_file = f"{base_dir}{location}.csv"
    humidity_file = f"{base_dir}humidity/{location}.csv"
    wind_file = f"{base_dir}wind/{location}.csv"

    # Check if files exist before loading
    try:
        df_temp = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(temp_file)
        df_humidity = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(humidity_file)
        df_wind = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(wind_file)

        # Perform an **inner join** on "Date"
        df_merged = df_temp.join(df_humidity, ["Date"], "inner").join(df_wind, ["Date"], "inner")

        # Save merged data
        merged_output_path = f"{base_dir}merged/{location}.csv"
        df_merged.write.format("csv").mode("overwrite").option("header", "true").save(merged_output_path)

        print(f"✅ Merged data saved: {merged_output_path}")

    except Exception as e:
        print(f"❌ Skipping {location} due to missing files: {e}")


In [0]:
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/PROENCA_A_NOVA.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SANTAREM.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/SAO_BRAS_DE_ALPORTEL.csv", recurse=True)
# dbutils.fs.rm("/mnt/weather_data/split_stations/merged/ALCACOVAS.csv", recurse=True)

In [0]:
from pyspark.sql.functions import col

# Define mapping for renaming columns
column_rename_map = {
    "Temperatura do ar m�dia di�ria (�C)": "Temperatura media do ar diaria (C)",
    "Precipita��o di�ria (mm)": "Precipitacao diaria (mm)",
    "Humidade relativa m�dia di�ria (%)": "Humidade relativa media diaria (%)",
    "Velocidade do vento m�dia di�ria (m/s)": "Velocidade do vento media diaria (m/s)"
}

# Define base directory
merged_dir = "/mnt/weather_data/split_stations/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    print(f"🔹 Processing file: {file_path}")

    # Read file
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Rename columns: Remove location name and apply clean headers
    new_columns = ["Date"] + [column_rename_map[col_name.split("_")[-1]] for col_name in df.columns[1:]]

    # Apply renaming
    df_cleaned = df.toDF(*new_columns)

    # Save back to CSV
    df_cleaned.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

    print(f"✅ Renamed and saved: {file_path}")

print("🎯 All merged tables have clean headers!")


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_stations/REBORDELO.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
stations_to_fix = ["REBORDELO", "VILA_NOVA_DE_CERVEIRA"]
input_dir = "/mnt/weather_data/split_stations/"

for station in stations_to_fix:
    path = f"{input_dir}{station}.csv"

    # Read CSV
    df = spark.read.option("header", True).csv(path)

    # Count total rows
    total_rows = df.count()

    # Add index
    df_indexed = df.rdd.zipWithIndex()

    # Filter: keep rows where index > 0 (drop first) and index < total_rows - 4 (drop last 4)
    df_filtered = df_indexed \
        .filter(lambda row: 0 < row[1] < total_rows - 4) \
        .map(lambda row: row[0])

    # Recreate DataFrame
    df_cleaned = spark.createDataFrame(df_filtered, schema=df.schema)

    # Save back to CSV (overwrite)
    df_cleaned.write.format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(path)

    print(f"✅ Cleaned and saved: {path} (dropped first + last 4 rows)")

In [0]:
dbutils.fs.rm("/mnt/weather_data/split_stations/PROENA-A-NOVA.csv", recurse=True)

In [0]:
display(dbutils.fs.ls("/mnt/weather_data/split_stations/"))


In [0]:
# Example: Read the "BATALHA.csv" file (replace with any filename you want to check)
file_to_check = "/mnt/weather_data/split_stations/REBORDELO.csv"

df_check = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_to_check)

# Display the contents of the file
df_check.display()

In [0]:
location_coordinates = {
    "REBORDELO/": (41.733, -7.167),
    "VILA_NOVA_DE_CERVEIRA/": (41.967, -8.683),
}


In [0]:
from pyspark.sql.functions import lit

# Define base directory
merged_dir = "/mnt/weather_data/split_stations/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name from filename

    # Check if we have coordinates for this location
    if location_name in location_coordinates:
        latitude, longitude = location_coordinates[location_name]

        print(f"🔹 Adding coordinates to {location_name}...")

        # Read dataset
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

        # Add Latitude and Longitude columns
        df_with_coords = df.withColumn("Latitude", lit(latitude)).withColumn("Longitude", lit(longitude))

        # Save updated dataset
        df_with_coords.write.format("csv").mode("overwrite").option("header", "true").save(file_path)

        print(f"✅ Updated and saved: {file_path}")
    else:
        print(f"⚠️ No coordinates found for {location_name}, skipping.")

print("🎯 All datasets now include coordinates!")


In [0]:
df_check = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/split_stations/REBORDELO.csv")
df_check.display()


In [0]:
# Function to clean column names for Delta tables
def clean_column_name(name):
    return (
        name.lower()  # Convert to lowercase
        .replace(" ", "_")  # Replace spaces with underscores
        .replace("(", "")  # Remove parentheses
        .replace(")", "")  # Remove parentheses
        .replace("%", "percent")  # Convert % to "percent"
        .replace(",", "")  # Remove commas
        .replace(";", "")  # Remove semicolons
    )

# Define base directories
merged_dir = "/mnt/weather_data/split_stations/"
delta_base_path = "/mnt/weather_data/delta/"

# Get list of merged files
merged_files = dbutils.fs.ls(merged_dir)

for file in merged_files:
    file_path = file.path
    location_name = file.name.replace(".csv", "")  # Extract location name

    print(f"🔹 Converting {location_name} to Delta format...")

    # Read CSV
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(file_path)

    # Clean column names
    new_columns = [clean_column_name(col_name) for col_name in df.columns]
    df_cleaned = df.toDF(*new_columns)

    # Define Delta table path
    delta_path = f"{delta_base_path}{location_name}"

    # Write to Delta format
    df_cleaned.write.format("delta").option("mergeSchema", "true").mode("overwrite").save(delta_path)

    print(f"✅ Delta table saved at: {delta_path}")

print("🎯 All datasets are now saved in Delta format with cleaned column names!")


In [0]:
display(dbutils.fs.ls("/mnt/weather_data/delta/"))

In [0]:
df_check = spark.read.format("delta").option("header", "true").option("inferSchema", "true").load("/mnt/weather_data/delta/REBORDELO/")
df_check.display()


In [0]:
import re

# Function to clean table names for Hive Metastore
def clean_table_name(name):
    name = name.rstrip("/")  # Remove trailing slashes
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)  # Replace invalid characters with "_"
    return name.lower()  # Convert to lowercase for consistency

# Define Hive table name (adjust schema/database if needed)
for file in dbutils.fs.ls(delta_base_path):
    location_name = clean_table_name(file.name)  # Clean table name

    print(f"🔹 Saving as Hive table: {location_name}...")

    # Read Delta table
    df = spark.read.format("delta").load(f"{delta_base_path}{file.name}")

    # Save as Hive table
    df.write.format("delta").mode("overwrite").saveAsTable(location_name)

    print(f"✅ Hive table created: {location_name}")

print("🎯 All Delta tables are now registered in the Hive Metastore!")


In [0]:
%sql

CREATE TABLE weather_data.rebordelo AS SELECT * FROM default.rebordelo;
DROP TABLE default.rebordelo;

CREATE TABLE weather_data.vila_nova_de_cerveira AS SELECT * FROM default.vila_nova_de_cerveira;
DROP TABLE default.vila_nova_de_cerveira;





In [0]:
%sql
CREATE DATABASE IF NOT EXISTS weather_data;